# Equipo 19 · Pesos del equipo anterior en AF-SfMLearner

Prueba los pesos del estimador de profundidad que dejó el equipo anterior y los compara con los de SCARED, que son
el punto de partida de todos nuestros modelos. Responde tres preguntas:

| Pregunta | Modelo | Sección |
|---|---|---|
| ¿Sus pesos, tal cual, estiman mejor la profundidad que los de SCARED? | **A′** frente a A | 5 |
| ¿Partir de sus pesos mejora el ajuste fino? | **C′, D′, E′** frente a C, D, E | 6 y 7 |
| ¿La ventaja del entrenamiento conjunto se mantiene si se cambia el punto de partida? | **E′ − D′** frente a E − D | 7 |

C′, D′ y E′ se entrenan **exactamente igual** que C, D y E en el Avance 5 (mismas funciones, datos, división, épocas
y semillas). La única diferencia es que el estimador parte de los pesos del equipo anterior.

Este notebook **no modifica** nada del Avance 5: lee sus datos y modelos y guarda todo lo nuevo en
`MyDrive/Equipo19/pesos_equipo_anterior_pruebas/`.

## 0. Configuración

**Colab Pro:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **A100 GPU** (o L4) y **RAM alta**.
Después: *Entorno de ejecución → Ejecutar todas*.

Revisa la celda siguiente:
- **`CARPETA_EQUIPO_ANTERIOR`**: la carpeta de Drive con el `encoder.pth` y el `depth.pth` que se quieren probar (por
  defecto, Monodepth2 entrenado en SCARED, extraído de `monodepth2_weights.zip`). La celda comprueba con una huella
  MD5 que no sean los mismos pesos SCARED que ya usamos; si lo son, se detiene.
- **`VIDEOS_ENTRENAMIENTO_ANTERIOR`**: los videos con los que el equipo anterior entrenó sus pesos. Sirve para
  comprobar que no vieron nuestros videos de prueba. Si no los conoces, déjalo vacío: el notebook avisará.
- **`SEMILLAS`** y **`ENTRENAR`**: qué modelos entrenar. Con una semilla y los tres modelos tarda alrededor de una
  hora en A100.

Las demás variables son las del Avance 5; no las cambies para que la comparación sea justa.

In [ ]:
from pathlib import Path
import os, sys, json, random, re, shutil, subprocess, time, zipfile, hashlib

# ---------- Ajusta aquí ----------
MYDRIVE = Path("/content/drive/MyDrive")
# Carpeta con encoder.pth y depth.pth del estimador a probar (Monodepth2 entrenado en SCARED, del trabajo anterior)
CARPETA_EQUIPO_ANTERIOR = MYDRIVE / "Equipo19" / "Pesos_equipo_anterior" / "monodepth2"
VIDEOS_ENTRENAMIENTO_ANTERIOR = []          # p. ej. ["c1_cecum_t1_a", ...]; vacío si no se conoce
SEMILLAS = [0]                               # [0, 1, 2] para comparar con las tres semillas del Avance 5
ENTRENAR = {"C": True, "D": True, "E": True}  # modelos que se entrenan partiendo de los pesos del equipo anterior
MASCARA = "meslouhi"                         # la máscara del modelo final

CARPETA_RAW = MYDRIVE / "Equipo19" / "raw"                       # ZIPs de C3VDv2
SALIDA = MYDRIVE / "Equipo19" / "end_to_end_mascaras"           # datos y modelos del Avance 5 (solo se leen)
SALIDA_ANTERIOR = MYDRIVE / "Equipo19" / "pesos_equipo_anterior_pruebas"   # todo lo nuevo de este notebook
CARPETAS_NUESTRAS = [SALIDA / "modelos"]    # aquí están C_s0..2, D_meslouhi_s0..2 y E_meslouhi_s0..2

# Lo mismo que el Avance 5: no cambiar, para que la comparación sea justa
EPOCAS = 15
PESOS_ENDO = {
    "meslouhi": MYDRIVE / "endo_corrida4" / "modelos" / "gen_mejor.pth",
    "arnold": MYDRIVE / "Equipo19" / "re_entrenar_endo" / "arnold_variaciones" / "A1" / "modelos" / "gen_mejor.pth",
}
DIVISION = {
    "entrenamiento": ["c2_cecum_t1_v1", "c2_rectum_t1_v1", "c2_transverse1_t1_v1", "c1_ascending_t3_v1"],
    "validacion": ["c2_sigmoid_t3_v2"],
    "prueba": ["c2_descending_t4_v2", "c2_descending_t4_v3"],
}
PASO, MAX_FRAMES = 5, 300
LOTE_D, LOTE_E = 8, 2
LR, LR_STTN = 1e-4, 1e-5
PESO_HUECO, PESO_ANCLA = 1.0, 0.0
STTN_CAPAS = "todo"
BLOQUE = 60

# Intervalos de confianza (como en la revisión del Avance 6)
N_BOOTSTRAP = 10000
LARGO_BLOQUE = 5
# ---------------------------------

SEMILLA = SEMILLAS[0]
OTRA = [m for m in PESOS_ENDO if m != MASCARA][0]
assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate", "scikit-image"], check=True)

LOCAL = Path("/content")
EXTERNOS = LOCAL / "externos"
DATOS = LOCAL / "datos_e2e"
MODELOS = SALIDA_ANTERIOR / "modelos"        # C′, D′ y E′ se guardan aquí, nunca en la carpeta del Avance 5
RES = SALIDA_ANTERIOR / "resultados"
for d in (EXTERNOS, DATOS, MODELOS, RES):
    d.mkdir(parents=True, exist_ok=True)

REPOS = {"Endo-STTN": ("https://github.com/endomapper/Endo-STTN.git", "9419a6c"),
         "AF-SfMLearner": ("https://github.com/ShuweiShao/AF-SfMLearner.git", "53d01a4")}
for nombre, (url, commit) in REPOS.items():
    if not (EXTERNOS / nombre).exists():
        subprocess.run(["git", "clone", "-q", url, str(EXTERNOS / nombre)], check=True)
        subprocess.run(["git", "-C", str(EXTERNOS / nombre), "checkout", "-q", commit], check=True)
    if str(EXTERNOS / nombre) not in sys.path:
        sys.path.append(str(EXTERNOS / nombre))

# Pesos oficiales de AF-SfMLearner en SCARED (Shao et al., 2022): el punto de partida de nuestros modelos.
# NO cambiar: aquí no van los pesos del trabajo anterior.
PESOS_SCARED = EXTERNOS / "pesos_scared"
if not (PESOS_SCARED / "encoder.pth").exists():
    z = MYDRIVE / "Equipo19" / "Model_trained_end_to_end.zip"
    if not z.is_file():
        import gdown
        z = EXTERNOS / "Model_trained_end_to_end.zip"
        gdown.download(id="1ZogZ5nQsfLhFAQGG_UE9A5R_osypLZf1", output=str(z), quiet=True)
    shutil.unpack_archive(str(z), str(EXTERNOS / "pesos_zip"))
    origen = next((EXTERNOS / "pesos_zip").rglob("encoder.pth")).parent
    PESOS_SCARED.mkdir(exist_ok=True)
    for f in ("encoder.pth", "depth.pth"):
        shutil.copy(origen / f, PESOS_SCARED / f)

# Solo el estimador (encoder.pth y depth.pth), copiado a una carpeta local
assert CARPETA_EQUIPO_ANTERIOR.is_dir(), f"No encuentro la carpeta {CARPETA_EQUIPO_ANTERIOR}"
PESOS_EQUIPO_ANTERIOR = LOCAL / "estimador_equipo_anterior"
shutil.rmtree(PESOS_EQUIPO_ANTERIOR, ignore_errors=True); PESOS_EQUIPO_ANTERIOR.mkdir()
for f in ("encoder.pth", "depth.pth"):
    assert (CARPETA_EQUIPO_ANTERIOR / f).is_file(), f"Falta {f} en {CARPETA_EQUIPO_ANTERIOR}"
    shutil.copy(CARPETA_EQUIPO_ANTERIOR / f, PESOS_EQUIPO_ANTERIOR / f)
for f in CARPETA_EQUIPO_ANTERIOR.glob("*"):
    if f.suffix.lower() in (".json", ".txt", ".md", ".yaml", ".yml"):
        shutil.copy(f, PESOS_EQUIPO_ANTERIOR / f.name)

# ¿Son distintos de los pesos SCARED? Si son iguales, A′ = A y la prueba no aporta nada.
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
IGUALES_A_SCARED = {f: md5(PESOS_EQUIPO_ANTERIOR / f) == md5(PESOS_SCARED / f) for f in ("encoder.pth", "depth.pth")}
for f, igual in IGUALES_A_SCARED.items():
    print(f"{f:12s}: {'IGUAL a SCARED' if igual else 'distinto de SCARED (pesos propios)'}")
assert not all(IGUALES_A_SCARED.values()), (
    "encoder.pth y depth.pth son IDÉNTICOS a los pesos SCARED oficiales: la prueba no aportaría nada. "
    "Pon en CARPETA_EQUIPO_ANTERIOR otro par encoder.pth/depth.pth.")

assert PESOS_ENDO[MASCARA].exists(), f"No encuentro el Endo-STTN del Avance 4: {PESOS_ENDO[MASCARA]}"
print("Pesos del equipo anterior:", CARPETA_EQUIPO_ANTERIOR, "->", PESOS_EQUIPO_ANTERIOR)
print("Pesos SCARED             :", PESOS_SCARED)
print("Resultados en            :", SALIDA_ANTERIOR)

## 1. Herramientas

Las cuatro celdas siguientes son **las mismas del notebook del Avance 5**, copiadas sin cambios: lectura de C3VDv2 y
máscaras, AF-SfMLearner, Endo-STTN y manejo de datos.

In [ ]:
import cv2
import numpy as np

ANCHO, ALTO = 320, 256       # entrada de AF-SfMLearner (C3VD es 5:4, igual que 320x256)
LADO = 288                   # entrada de Endo-STTN
PROF_MAX_MM = 100.0


def indice(nombre):
    """Número de frame dentro del nombre de archivo ('0042_color.png' -> 42)."""
    m = re.search(r"(\d+)", Path(nombre).stem)
    return int(m.group(1)) if m else None


def listar_pares(zf):
    """Pares (indice, ruta_rgb, ruta_depth) presentes en las dos carpetas, ordenados."""
    nombres = zf.namelist()
    rgb = {indice(n): n for n in nombres if n.startswith("rgb/") and n.lower().endswith((".png", ".jpg"))}
    dep = {indice(n): n for n in nombres if n.startswith("depth/") and n.lower().endswith((".tiff", ".tif", ".png"))}
    if not rgb or not dep:
        raise ValueError(f"El ZIP no tiene rgb/ y depth/ (carpetas: {sorted({n.split('/')[0] for n in nombres})})")
    return [(i, rgb[i], dep[i]) for i in sorted(set(rgb) & set(dep) - {None})]


def leer_rgb(zf, ruta):
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def leer_profundidad(zf, ruta):
    """Profundidad en mm (float32). 0 = sin dato (incluye píxeles saturados)."""
    d = cv2.imdecode(np.frombuffer(zf.read(ruta), np.uint8), cv2.IMREAD_UNCHANGED)
    d = d[..., 0] if d.ndim == 3 else d
    if d.dtype == np.uint16:
        mm = d.astype(np.float32) * (PROF_MAX_MM / 65535.0)
        mm[d == 65535] = 0
        return mm
    return d.astype(np.float32)


def _srgb_a_xyz(img):
    lin = np.where(img <= 0.04045, img / 12.92, ((img + 0.055) / 1.055) ** 2.4)
    m = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    return lin @ m.T


def _dilatar_1px(m):
    return cv2.dilate(m.astype(np.uint8), np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)).astype(bool)


def mascara_meslouhi(rgb):
    """El Meslouhi et al. (2011), la de los autores de Endo-STTN: realce por min/max, CIE-XYZ,
    reflejo donde Y >= Y/(X+Y+Z), dilatación de 1 px."""
    img = rgb.astype(np.float64) / 255
    mn, mx = img.min(-1), img.max(-1)
    s = np.divide(mn, mx, out=np.zeros_like(mn), where=mx > 0)
    xyz = _srgb_a_xyz(s[..., None] * img)
    suma = xyz.sum(-1)
    y = np.divide(xyz[..., 1], suma, out=np.full_like(suma, np.inf), where=suma > 0)
    return _dilatar_1px(xyz[..., 1] >= y)


def mascara_arnold(rgb, t_abs=240, t_cand=200, t_rel=1.2, mediana=31):
    """Arnold et al. (2010), versión simplificada: detector absoluto (muy brillante, corregido por el balance
    de color) + detector relativo (más brillante que una imagen "sin reflejos" estimada), dilatación de 1 px."""
    img = rgb.astype(np.float32)
    gris = img @ np.array([0.2989, 0.5870, 0.1140], np.float32)
    p95 = lambda c: max(float(np.percentile(c, 95)), 1.0)
    r_g, r_b = p95(img[..., 1]) / p95(gris), p95(img[..., 2]) / p95(gris)
    sobre = lambda t: (img[..., 1] > r_g * t) | (img[..., 2] > r_b * t) | (gris > t)
    candidatos = cv2.dilate(sobre(t_cand).astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    sin_reflejos = cv2.medianBlur(cv2.inpaint(rgb, candidatos, 5, cv2.INPAINT_TELEA), mediana).astype(np.float32)
    relativo = ((img / (sin_reflejos + 1)).max(-1) > t_rel) & (gris > t_cand * 0.75)
    return _dilatar_1px(sobre(t_abs) | relativo)


METODOS = {"meslouhi": mascara_meslouhi, "arnold": mascara_arnold}


def mascara(rgb_320, metodo):
    """Máscara a 320x256 calculada sobre el frame a 288x288, como al reentrenar Endo-STTN."""
    m = METODOS[metodo](cv2.resize(rgb_320, (LADO, LADO), interpolation=cv2.INTER_AREA))
    return cv2.resize(m.astype(np.uint8), (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST).astype(bool)

In [ ]:
import warnings
import torch
import torch.nn.functional as F
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")   # aviso de 'pretrained' en AF-SfMLearner
import networks                                  # AF-SfMLearner
from model.sttn import InpaintGenerator          # Endo-STTN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROF_MIN, PROF_MAX = 0.1, PROF_MAX_MM


def cargar_estimador(carpeta):
    enc = networks.ResnetEncoder(18, False)
    sd = torch.load(Path(carpeta) / "encoder.pth", map_location=DEVICE)
    enc.load_state_dict({k: v for k, v in sd.items() if k in enc.state_dict()})
    dec = networks.DepthDecoder(num_ch_enc=enc.num_ch_enc, scales=range(4))
    dec.load_state_dict(torch.load(Path(carpeta) / "depth.pth", map_location=DEVICE))
    return enc.to(DEVICE), dec.to(DEVICE)


def guardar_estimador(enc, dec, carpeta):
    carpeta.mkdir(parents=True, exist_ok=True)
    sd = enc.state_dict()
    sd.update(height=ALTO, width=ANCHO)
    torch.save(sd, carpeta / "encoder.pth")
    torch.save(dec.state_dict(), carpeta / "depth.pth")


def a_tensor(rgb_uint8):
    """(B,H,W,3) uint8 -> (B,3,H,W) en [0,1]."""
    return torch.from_numpy(np.array(rgb_uint8)).permute(0, 3, 1, 2).float().div(255).to(DEVICE)


def predecir(enc, dec, x):
    """Profundidad en mm, (B,1,H,W). Igual que disp_to_depth de Monodepth2/AF-SfMLearner."""
    disp = dec(enc(x))[("disp", 0)]
    min_disp, max_disp = 1 / PROF_MAX, 1 / PROF_MIN
    return 1 / (min_disp + (max_disp - min_disp) * disp)


def perdida_si_log(pred, gt):
    """Scale-invariant log por imagen, sobre los píxeles con profundidad real."""
    perdidas = []
    for p, g in zip(pred, gt):
        valido = g > 0
        if valido.sum() < 10:
            continue
        d = torch.log(p[valido]) - torch.log(g[valido])
        perdidas.append((d ** 2).mean() - d.mean() ** 2)
    return torch.stack(perdidas).mean()


def metricas(pred, gt, region=None):
    """Métricas con median scaling. La escala se calcula con toda la imagen y luego se mide en la región."""
    valido = (gt > 0) & (gt <= PROF_MAX)
    if valido.sum() < 10:
        return None
    escala = np.median(gt[valido]) / np.median(pred[valido])
    if region is not None:
        valido &= region
        if valido.sum() < 10:
            return None
    p, g = pred[valido].astype(np.float64), gt[valido].astype(np.float64)
    p = np.clip(p * escala, PROF_MIN, PROF_MAX)
    ratio = np.maximum(p / g, g / p)
    return {"abs_rel": float(np.mean(np.abs(p - g) / g)), "sq_rel": float(np.mean((p - g) ** 2 / g)),
            "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
            "rmse_log": float(np.sqrt(np.mean((np.log(p) - np.log(g)) ** 2))), "a1": float(np.mean(ratio < 1.25))}


@torch.no_grad()
def predecir_video(enc, dec, rgb):
    enc.eval(); dec.eval()
    return np.concatenate([predecir(enc, dec, a_tensor(rgb[i:i + 16]))[:, 0].cpu().numpy()
                           for i in range(0, len(rgb), 16)])

In [ ]:
NEIGHBOR_STRIDE, REF_LENGTH = 5, 10   # como en Endo-STTN/test.py
DESPLAZAMIENTO = 56


def cargar_endo(pesos):
    sd = torch.load(pesos, map_location=DEVICE)
    sd = {k.removeprefix("module."): v for k, v in sd.get("netG", sd).items()}
    modelo = InpaintGenerator().to(DEVICE)
    modelo.load_state_dict(sd)
    return modelo.eval()


@torch.no_grad()
def limpiar_bloque(modelo, rgb, mask):
    """rgb (T,H,W,3) uint8, mask (T,H,W) bool -> (T,H,W,3) uint8 con los reflejos rellenados."""
    T, H, W = mask.shape
    f = np.stack([cv2.resize(x, (LADO, LADO), interpolation=cv2.INTER_AREA) for x in rgb])
    m = np.stack([cv2.resize(x.astype(np.uint8), (LADO, LADO), interpolation=cv2.INTER_NEAREST) for x in mask])
    feats = torch.from_numpy(f).permute(0, 3, 1, 2).float().div(255).mul(2).sub(1).to(DEVICE)   # [-1,1]
    masks = torch.from_numpy(m).float().unsqueeze(1).to(DEVICE)
    enc = modelo.encoder(feats * (1 - masks))
    _, c, fh, fw = enc.shape
    enc = enc.view(1, T, c, fh, fw)
    acum, cuenta = np.zeros((T, LADO, LADO, 3), np.float32), np.zeros(T, np.float32)
    for k in range(0, T, NEIGHBOR_STRIDE):
        vec = list(range(max(0, k - NEIGHBOR_STRIDE), min(T, k + NEIGHBOR_STRIDE + 1)))
        ids = vec + [i for i in range(0, T, REF_LENGTH) if i not in vec]
        pred = torch.tanh(modelo.decoder(modelo.infer(enc[0, ids], masks[ids])[:len(vec)]))
        pred = ((pred + 1) / 2).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy() * 255
        for i, idx in enumerate(vec):
            acum[idx] += pred[i]
            cuenta[idx] += 1
    pred = acum / cuenta[:, None, None, None]
    out = rgb.copy()
    for t in range(T):
        relleno = cv2.resize(pred[t], (W, H), interpolation=cv2.INTER_LINEAR)
        out[t][mask[t]] = np.clip(relleno[mask[t]], 0, 255).astype(np.uint8)
    return out


def limpiar_video(modelo, rgb, mask):
    rgb, mask = np.asarray(rgb), np.asarray(mask)
    return np.concatenate([limpiar_bloque(modelo, rgb[i:i + BLOQUE], mask[i:i + BLOQUE])
                           for i in range(0, len(rgb), BLOQUE)])


def desplazar(m, dx=DESPLAZAMIENTO):
    """Máscara desplazada dx px a la derecha."""
    out = torch.zeros_like(m)
    out[..., dx:] = m[..., :-dx]
    return out


def rellenar(sttn, x, hueco):
    """Relleno diferenciable. x (B,T,3,H,W) en [0,1], hueco (B,T,1,H,W) -> (B,T,3,H,W)."""
    B, T, _, H, W = x.shape
    f = F.interpolate(x.flatten(0, 1), (LADO, LADO), mode="area") * 2 - 1
    m = F.interpolate(hueco.flatten(0, 1), (LADO, LADO), mode="nearest")
    out = sttn((f * (1 - m)).view(B, T, 3, LADO, LADO), m.view(B, T, 1, LADO, LADO))
    out = F.interpolate((out + 1) / 2, (H, W), mode="bilinear", align_corners=False)
    return out.view(B, T, 3, H, W)


def l1_en(a, b, region):
    """L1 media dentro de una región (B,T,1,H,W); 0 si la región está vacía."""
    return ((a - b).abs() * region).sum() / (region.sum() * a.shape[2]).clamp(min=1)

In [ ]:
def ruta(video, nombre):
    return DATOS / video / f"{nombre}.npy"


def existe(video, nombre):
    """¿Ya existe? Si solo está en Drive (de otra sesión), lo copia a local."""
    en_drive = SALIDA / "datos" / video / f"{nombre}.npy"
    if not ruta(video, nombre).exists() and en_drive.exists():
        ruta(video, nombre).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(en_drive, ruta(video, nombre))
    return ruta(video, nombre).exists()


def guardar(video, nombre, arr):
    """Guarda en local y en Drive (primero con otro nombre, para no dejar archivos a medias)."""
    ruta(video, nombre).parent.mkdir(parents=True, exist_ok=True)
    np.save(ruta(video, nombre), arr)
    destino = SALIDA / "datos" / video
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy(ruta(video, nombre), destino / f"{nombre}.tmp")
    os.replace(destino / f"{nombre}.tmp", destino / f"{nombre}.npy")


def cargar(video, nombre):
    assert existe(video, nombre), f"Falta {nombre}.npy de {video}"
    return np.load(ruta(video, nombre), mmap_mode="r")


class Frames(torch.utils.data.Dataset):
    def __init__(self, videos, entrada):
        self.datos = [(cargar(v, entrada), cargar(v, "depth")) for v in videos]
        self.indices = [(k, i) for k, (rgb, _) in enumerate(self.datos) for i in range(len(rgb))]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k, i = self.indices[n]
        rgb = np.array(self.datos[k][0][i], np.float32) / 255
        prof = np.array(self.datos[k][1][i], np.float32)
        if random.random() < 0.5:
            rgb, prof = rgb[:, ::-1], prof[:, ::-1]
        rgb = np.clip(rgb * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05), 0, 1)
        return torch.from_numpy(rgb.transpose(2, 0, 1).copy()), torch.from_numpy(prof.copy())[None]


class Clips(torch.utils.data.Dataset):
    def __init__(self, videos, mascara, T=5):
        self.datos = [(cargar(v, "rgb"), cargar(v, "depth"), cargar(v, f"mask_{mascara}"),
                       cargar(v, f"rgb_limpio_{mascara}")) for v in videos]
        self.T = T
        self.indices = [(k, i) for k, d in enumerate(self.datos) for i in range(0, len(d[0]) - T + 1, T)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k, i = self.indices[n]
        rgb, prof, mask, limpio = self.datos[k]
        ids = sorted(random.sample(range(len(rgb)), self.T)) if random.random() < 0.5 else list(range(i, i + self.T))
        x = np.stack([rgb[j] for j in ids]).astype(np.float32) / 255
        lim = np.stack([limpio[j] for j in ids]).astype(np.float32) / 255
        m = np.stack([mask[j] for j in ids]).astype(np.float32)
        g = np.stack([prof[j] for j in ids]).astype(np.float32)
        if random.random() < 0.5:
            x, lim, m, g = x[:, :, ::-1], lim[:, :, ::-1], m[:, :, ::-1], g[:, :, ::-1]
        brillo = np.array([random.uniform(0.9, 1.1), random.uniform(-0.05, 0.05)], np.float32)
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a))
        return t(x.transpose(0, 3, 1, 2)), t(lim.transpose(0, 3, 1, 2)), t(m[:, None]), t(g[:, None]), t(brillo)


def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    return torch.Generator().manual_seed(s)


print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (sin GPU: será muy lento)")

## 2. ¿Qué hay en los pesos del equipo anterior?

La celda abre la carpeta (o el `.zip`), lee cada archivo de pesos y lo compara, tensor por tensor, con las tres redes
que usamos:

| Red | Archivo esperado |
|---|---|
| Codificador de AF-SfMLearner (ResNet-18 o ResNet-50) | `encoder.pth` |
| Decodificador de profundidad de AF-SfMLearner | `depth.pth` |
| Endo-STTN (opcional) | `gen_*.pth` o `sttn.pth` |

Un archivo se considera compatible si al menos el 95 % de los tensores de la red coinciden en nombre y tamaño. Los
compatibles se copian a `/content/pesos_anterior/` con los nombres que usa nuestro código. También se muestran los
archivos de texto que traiga la carpeta (por ejemplo `opt.json`), donde suele estar con qué datos entrenaron.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

CRUDO = LOCAL / "pesos_anterior_crudo"
PESOS_ANT = LOCAL / "pesos_anterior"
for d in (CRUDO, PESOS_ANT):
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
origen = Path(PESOS_EQUIPO_ANTERIOR)
if origen.is_dir():
    shutil.copytree(origen, CRUDO, dirs_exist_ok=True)
elif origen.suffix.lower() == ".zip":
    shutil.unpack_archive(str(origen), str(CRUDO))
else:
    shutil.copy(origen, CRUDO / origen.name)
for z in list(CRUDO.rglob("*.zip")):                     # zips dentro de la carpeta
    shutil.unpack_archive(str(z), str(z.with_suffix("")))


def aplanar(obj, prefijo=""):
    """{nombre: tensor} de un state_dict, también si viene dentro de otro diccionario (p. ej. {"netG": ...})."""
    salida = {}
    if isinstance(obj, torch.nn.Module):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if torch.is_tensor(v):
                salida[prefijo + str(k)] = v
            elif isinstance(v, (dict, torch.nn.Module)):
                salida.update(aplanar(v, f"{prefijo}{k}."))
    return salida


def ajustar(sd, ref):
    """Busca el prefijo que hay que quitar para que los nombres coincidan con la red. Devuelve (fracción, state_dict)."""
    prefijos = {""} | {k[:i + 1] for k in sd for i in range(len(k)) if k[i] == "."}
    mejor = (0.0, {})
    for p in sorted(prefijos, key=len):
        sub = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
        iguales = {k: v for k, v in sub.items() if k in ref and tuple(v.shape) == tuple(ref[k].shape)}
        frac = len(iguales) / len(ref)
        if frac > mejor[0]:
            mejor = (frac, iguales)
    return mejor


REDES = {"codificador ResNet-18": networks.ResnetEncoder(18, False).state_dict(),
         "codificador ResNet-50": networks.ResnetEncoder(50, False).state_dict(),
         "Endo-STTN": InpaintGenerator().state_dict()}
for capas, enc_ref in ((18, networks.ResnetEncoder(18, False)), (50, networks.ResnetEncoder(50, False))):
    REDES[f"decodificador (ResNet-{capas})"] = networks.DepthDecoder(num_ch_enc=enc_ref.num_ch_enc, scales=range(4)).state_dict()

filas, ENCONTRADO = [], {}
for f in sorted(CRUDO.rglob("*")):
    if f.suffix.lower() not in (".pth", ".pt", ".ckpt", ".tar", ".bin"):
        continue
    try:
        crudo = torch.load(f, map_location="cpu", weights_only=False)
        sd = aplanar(crudo)
    except Exception as e:
        filas.append({"archivo": str(f.relative_to(CRUDO)), "tensores": 0, "red": f"no se pudo leer ({type(e).__name__})", "coincidencia": ""})
        continue
    resultados = {red: ajustar(sd, ref) for red, ref in REDES.items()}
    red, (frac, ajustado) = max(resultados.items(), key=lambda x: x[1][0])
    filas.append({"archivo": str(f.relative_to(CRUDO)), "tensores": len(sd),
                  "red": red if frac >= 0.95 else "no compatible", "coincidencia": f"{100 * frac:.0f} %"})
    if frac >= 0.95 and red not in ENCONTRADO:
        ENCONTRADO[red] = (f, ajustado)
    if isinstance(crudo, dict) and "height" in crudo and "width" in crudo:
        print(f"{f.relative_to(CRUDO)}: entrenado a {crudo['width']}x{crudo['height']} (nuestro código usa {ANCHO}x{ALTO})")

display(Markdown("**Archivos de pesos encontrados**"))
display(pd.DataFrame(filas).set_index("archivo") if filas else Markdown("No hay archivos de pesos."))

for f in sorted(CRUDO.rglob("*")):
    if f.suffix.lower() in (".json", ".txt", ".md", ".yaml", ".yml", ".csv") and f.stat().st_size < 200_000:
        texto = f.read_text(errors="ignore")
        display(Markdown(f"**{f.relative_to(CRUDO)}**\n```\n{texto[:3000]}\n```"))

capas = 50 if "codificador ResNet-50" in ENCONTRADO and "codificador ResNet-18" not in ENCONTRADO else 18
codificador, decodificador = ENCONTRADO.get(f"codificador ResNet-{capas}"), ENCONTRADO.get(f"decodificador (ResNet-{capas})")
assert codificador and decodificador, (
    "No encontré un codificador y un decodificador de AF-SfMLearner compatibles en los pesos del equipo anterior. "
    "Revisa la tabla: si su modelo es otra arquitectura (DPT, Depth Anything, etc.), este notebook no lo puede usar.")
torch.save(codificador[1], PESOS_ANT / "encoder.pth")
torch.save(decodificador[1], PESOS_ANT / "depth.pth")
json.dump({"capas": capas, "origen": {"codificador": str(codificador[0]), "decodificador": str(decodificador[0])}},
          open(PESOS_ANT / "capas.json", "w"), indent=2)
STTN_ANT = None
if "Endo-STTN" in ENCONTRADO:
    STTN_ANT = PESOS_ANT / "sttn.pth"
    torch.save({"netG": ENCONTRADO["Endo-STTN"][1]}, STTN_ANT)
print(f"\nEstimador del equipo anterior: AF-SfMLearner con ResNet-{capas}")
print("  codificador  :", codificador[0].relative_to(CRUDO))
print("  decodificador:", decodificador[0].relative_to(CRUDO))
print("Endo-STTN del equipo anterior:", ENCONTRADO["Endo-STTN"][0].relative_to(CRUDO) if STTN_ANT else "no viene")

In [ ]:
# Nuestro código carga siempre ResNet-18. Estas versiones leen capas.json para aceptar también ResNet-50.
def cargar_estimador(carpeta):
    carpeta = Path(carpeta)
    capas = json.load(open(carpeta / "capas.json"))["capas"] if (carpeta / "capas.json").exists() else 18
    enc = networks.ResnetEncoder(capas, False)
    sd = torch.load(carpeta / "encoder.pth", map_location=DEVICE)
    enc.load_state_dict({k: v for k, v in sd.items() if k in enc.state_dict()})
    dec = networks.DepthDecoder(num_ch_enc=enc.num_ch_enc, scales=range(4))
    dec.load_state_dict(torch.load(carpeta / "depth.pth", map_location=DEVICE))
    return enc.to(DEVICE), dec.to(DEVICE)


def guardar_estimador(enc, dec, carpeta):
    carpeta.mkdir(parents=True, exist_ok=True)
    sd = enc.state_dict()
    sd.update(height=ALTO, width=ANCHO)
    torch.save(sd, carpeta / "encoder.pth")
    torch.save(dec.state_dict(), carpeta / "depth.pth")
    json.dump({"capas": 50 if enc.num_ch_enc[-1] == 2048 else 18}, open(carpeta / "capas.json", "w"))


enc, dec = cargar_estimador(PESOS_ANT)      # comprobación: carga sin errores
print("Los pesos del equipo anterior cargan en AF-SfMLearner.")
del enc, dec

## 3. Datos

Los mismos datos del Avance 5 (`end_to_end_mascaras/datos/`). Si faltan, se preparan con la misma configuración, y
los frames limpios con el Endo-STTN congelado se calculan una sola vez (los usan D′ y E′). Son las mismas celdas del
Avance 5.

In [ ]:
VIDEOS = [v for g in DIVISION.values() for v in g]
prep = {"paso": PASO, "max_frames": MAX_FRAMES, "division": DIVISION}
f_prep = SALIDA / "preparacion.json"
if f_prep.exists():
    assert json.load(open(f_prep)) == prep, (
        f"Los datos de {SALIDA} se prepararon con otra configuración: {json.load(open(f_prep))}. "
        "Regresa PASO, MAX_FRAMES y DIVISION a esos valores, o usa otra carpeta SALIDA.")
else:
    json.dump(prep, open(f_prep, "w"), indent=2)


def zip_de(video):
    """ZIP de un video en CARPETA_RAW: nombre exacto, o con sufijo de Drive ("-007", " (1)")."""
    if (CARPETA_RAW / f"{video}.zip").exists():
        return CARPETA_RAW / f"{video}.zip"
    candidatos = [z for z in CARPETA_RAW.glob(f"{video}*.zip") if not z.stem[len(video)].isalnum()
                  and z.stem[len(video)] != "_"]
    assert len(candidatos) == 1, f"Para {video} hay {len(candidatos)} ZIPs en {CARPETA_RAW}: {candidatos}"
    return candidatos[0]


for k, v in enumerate(VIDEOS, 1):
    if all(existe(v, n) for n in ["rgb", "depth", "mask_meslouhi", "mask_arnold"]):
        print(f"[{k}/{len(VIDEOS)}] {v}: ya preparado")
        continue
    with zipfile.ZipFile(zip_de(v)) as zf:
        pares = listar_pares(zf)[::PASO][:MAX_FRAMES or None]
        rgbs, profs = [], []
        for _, r, d in pares:
            rgbs.append(cv2.resize(leer_rgb(zf, r), (ANCHO, ALTO), interpolation=cv2.INTER_AREA))
            profs.append(cv2.resize(leer_profundidad(zf, d), (ANCHO, ALTO),
                                    interpolation=cv2.INTER_NEAREST).astype(np.float16))
    rgb = np.stack(rgbs)
    guardar(v, "rgb", rgb)
    guardar(v, "depth", np.stack(profs))
    for m in METODOS:
        guardar(v, f"mask_{m}", np.stack([mascara(x, m) for x in rgb]))
    print(f"[{k}/{len(VIDEOS)}] {v}: {len(rgb)} frames")

import pandas as pd
from IPython.display import Markdown, display
grupo = {v: g for g, vs in DIVISION.items() for v in vs}
display(pd.DataFrame([{"video": v, "grupo": grupo[v], "frames": len(cargar(v, "rgb")),
                       **{f"% reflejos {m}": round(100 * float(np.mean(cargar(v, f"mask_{m}"))), 2) for m in METODOS}}
                      for v in VIDEOS]).set_index("video"))

In [ ]:
endo = cargar_endo(PESOS_ENDO[MASCARA])
for k, v in enumerate(VIDEOS, 1):
    if existe(v, f"rgb_limpio_{MASCARA}"):
        print(f"[{k}/{len(VIDEOS)}] {v}: ya limpio")
        continue
    t0 = time.time()
    guardar(v, f"rgb_limpio_{MASCARA}", limpiar_video(endo, cargar(v, "rgb"), cargar(v, f"mask_{MASCARA}")))
    print(f"[{k}/{len(VIDEOS)}] {v}: limpio ({time.time() - t0:.0f} s)")
del endo; torch.cuda.empty_cache()

## 4. ¿Los pesos del equipo anterior vieron nuestros videos de prueba?

Si el equipo anterior entrenó con videos de C3VD o C3VDv2, hay que comprobar que **ningún video ni segmento** de
nuestra prueba (`c2_descending_t4`) estuvo en su entrenamiento. Si lo estuvo, sus resultados en nuestra prueba
estarían inflados por **fuga de datos por grupo**.

La comprobación usa `VIDEOS_ENTRENAMIENTO_ANTERIOR` y, además, busca nombres de videos en los archivos de texto de su
carpeta.

In [ ]:
base_video = lambda v: (re.match(r"(c\d+_[a-z0-9]+_t\d+_v\d+)", v) or [v])[0]
segmento = lambda v: re.sub(r"_v\d+$", "", base_video(v))

en_textos = set()
for f in CRUDO.rglob("*"):
    if f.is_file() and f.suffix.lower() in (".json", ".txt", ".md", ".yaml", ".yml", ".csv") and f.stat().st_size < 5_000_000:
        en_textos |= set(re.findall(r"c\d+_[a-z0-9]+_t\d+_v\d+", f.read_text(errors="ignore")))
videos_ant = {base_video(v) for v in VIDEOS_ENTRENAMIENTO_ANTERIOR} | en_textos
prueba = {base_video(v) for v in DIVISION["prueba"]}
seg_prueba = {segmento(v) for v in prueba}

if not videos_ant:
    FUGA_ANTERIOR = "desconocida"
    display(Markdown("**AVISO:** no se conoce con qué videos entrenó el equipo anterior. Los resultados de A′ y de "
                     "los modelos que parten de sus pesos deben reportarse con esta advertencia. Pide su lista de "
                     "videos y ponla en `VIDEOS_ENTRENAMIENTO_ANTERIOR`."))
else:
    mismo_video = sorted(videos_ant & prueba)
    mismo_segmento = sorted(v for v in videos_ant if segmento(v) in seg_prueba)
    FUGA_ANTERIOR = "sí" if mismo_segmento else "no"
    display(pd.DataFrame([{"videos conocidos del equipo anterior": len(videos_ant),
                           "mismo video que nuestra prueba": ", ".join(mismo_video) or "ninguno",
                           "mismo segmento que nuestra prueba": ", ".join(mismo_segmento) or "ninguno"}]))
    display(Markdown("**FUGA:** el equipo anterior entrenó con nuestro segmento de prueba; sus resultados en esta prueba "
                     "no son válidos." if mismo_segmento else "**Sin fuga** con los videos conocidos del equipo anterior."))

## 5. Los pesos tal cual: A′ frente a A

Antes de entrenar nada: el estimador del equipo anterior, **sin ajustar**, sobre los frames originales de prueba,
frente al de SCARED (nuestro modelo A). La tabla con intervalos de confianza está en la sección 7.

In [ ]:
import matplotlib.pyplot as plt

VIDEOS_PRUEBA = DIVISION["prueba"]
UNION = {v: np.asarray(cargar(v, "mask_meslouhi")) | np.asarray(cargar(v, "mask_arnold")) for v in VIDEOS_PRUEBA}
pred_sin_ajustar = {}
for nombre, carpeta in (("A · SCARED", PESOS_SCARED), ("A′ · equipo anterior", PESOS_ANT)):
    enc, dec = cargar_estimador(carpeta)
    pred_sin_ajustar[nombre] = {v: predecir_video(enc, dec, cargar(v, "rgb")) for v in VIDEOS_PRUEBA}
    del enc, dec

v = VIDEOS_PRUEBA[0]
rgb, prof = cargar(v, "rgb"), cargar(v, "depth")
elegidos = np.linspace(0, len(rgb) - 1, 3).astype(int)
fig, ax = plt.subplots(len(elegidos), 4, figsize=(15, 3.2 * len(elegidos)), squeeze=False)
for r, i in enumerate(elegidos):
    g = np.asarray(prof[i], np.float32); valido = g > 0
    vmin, vmax = np.percentile(g[valido], [2, 98])
    ax[r, 0].imshow(rgb[i]); ax[r, 0].set_title(f"{v} · frame {i}", fontsize=9)
    ax[r, 1].imshow(np.where(valido, g, np.nan), cmap="magma_r", vmin=vmin, vmax=vmax); ax[r, 1].set_title("profundidad real", fontsize=9)
    for k, (nombre, preds) in enumerate(pred_sin_ajustar.items(), 2):
        p = preds[v][i] * np.median(g[valido]) / np.median(preds[v][i][valido])
        m = metricas(preds[v][i], g)
        ax[r, k].imshow(np.where(valido, p, np.nan), cmap="magma_r", vmin=vmin, vmax=vmax)
        ax[r, k].set_title(f"{nombre}\nAbsRel {m['abs_rel']:.3f}" if m else nombre, fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.suptitle("Estimadores sin ajustar sobre frames originales (escalados con la mediana de la real)", y=1.0)
plt.tight_layout(); plt.savefig(RES / "sin_ajustar_A_vs_A_prima.png", dpi=90, bbox_inches="tight"); plt.show()

## 6. Ajuste fino partiendo de los pesos del equipo anterior: C′, D′ y E′

Las funciones de entrenamiento son **las mismas del Avance 5**. Solo cambian dos cosas:
- el estimador parte de `PESOS_BASE` (los pesos del equipo anterior) en lugar de SCARED;
- los modelos se guardan en `pesos_equipo_anterior_pruebas/modelos/`, y los frames limpiados por E′ con otro nombre,
  para no pisar nada del Avance 5.

| Modelo | Endo-STTN | Estimador |
|---|---|---|
| C′ | ninguno | parte del equipo anterior; frames originales |
| D′ | reentrenado y congelado (Avance 4) | parte del equipo anterior; frames limpios |
| E′ | reentrenado; sigue aprendiendo | parte del equipo anterior; entrenamiento conjunto |

Si se interrumpe la sesión, vuelve a ejecutar todo: los modelos terminados se reutilizan.

In [ ]:
PESOS_BASE = PESOS_ANT   # punto de partida del estimador en C′, D′ y E′

In [ ]:
@torch.no_grad()
def validar_estimador(enc, dec, entrada):
    vals = []
    for v in DIVISION["validacion"]:
        pred, prof = predecir_video(enc, dec, cargar(v, entrada)), cargar(v, "depth")
        vals += [r["abs_rel"] for p, g in zip(pred, prof) if (r := metricas(p, np.asarray(g, np.float32)))]
    return float(np.mean(vals))


def terminado(nombre):
    f = MODELOS / nombre / "historial.json"
    return f.exists() and "mejor_abs_rel_val" in json.load(open(f))


def entrenar_estimador(nombre, entrada):
    if terminado(nombre):
        print(f"{nombre}: ya entrenado, se reutiliza"); return
    salida = MODELOS / nombre
    gen = fijar_semilla(SEMILLA)
    datos = Frames(DIVISION["entrenamiento"], entrada)
    carga = torch.utils.data.DataLoader(datos, batch_size=LOTE_D, shuffle=True, drop_last=True, generator=gen)
    enc, dec = cargar_estimador(PESOS_BASE)
    opt = torch.optim.Adam([{"params": enc.parameters(), "lr": LR / 10}, {"params": dec.parameters(), "lr": LR}])
    historial = {"entrada": entrada, "semilla": SEMILLA, "division": DIVISION, "epocas": []}
    mejor = validar_estimador(enc, dec, entrada)
    historial["epocas"].append({"epoca": 0, "abs_rel_val": mejor})
    guardar_estimador(enc, dec, salida)
    print(f"{nombre} ({len(datos)} frames) · época 0 (pesos de inicio): AbsRel validación {mejor:.4f}")
    for ep in range(1, EPOCAS + 1):
        t0, perdidas = time.time(), []
        enc.train(); dec.train()
        for x, gt in carga:
            loss = perdida_si_log(predecir(enc, dec, x.to(DEVICE)), gt.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            perdidas.append(loss.item())
        val = validar_estimador(enc, dec, entrada)
        historial["epocas"].append({"epoca": ep, "perdida": float(np.mean(perdidas)), "abs_rel_val": val})
        marca = ""
        if val < mejor:
            mejor, marca = val, "  <- mejor, guardado"
            guardar_estimador(enc, dec, salida)
        print(f"  época {ep}: pérdida {np.mean(perdidas):.4f} | AbsRel validación {val:.4f} | {time.time() - t0:.0f} s{marca}")
        json.dump(historial, open(salida / "historial.json", "w"), indent=2)
    historial["mejor_abs_rel_val"] = mejor
    json.dump(historial, open(salida / "historial.json", "w"), indent=2)

In [ ]:
@torch.no_grad()
def validar_conjunto(sttn, enc, dec):
    """AbsRel de validación y cambio medio del relleno respecto al Endo-STTN congelado (0-255)."""
    sttn.eval()
    absrel, cambio = [], []
    for v in DIVISION["validacion"]:
        mask = np.asarray(cargar(v, f"mask_{MASCARA}"))
        limpio = limpiar_video(sttn, cargar(v, "rgb"), mask)
        if mask.any():
            cambio.append(np.abs(limpio.astype(np.int16) - cargar(v, f"rgb_limpio_{MASCARA}"))[mask].mean())
        pred = predecir_video(enc, dec, limpio)
        absrel += [r["abs_rel"] for p, g in zip(pred, cargar(v, "depth")) if (r := metricas(p, np.asarray(g, np.float32)))]
    return float(np.mean(absrel)), float(np.mean(cambio)) if cambio else float("nan")


def guardar_conjunto(sttn, enc, dec, carpeta):
    guardar_estimador(enc, dec, carpeta)
    torch.save({"netG": sttn.state_dict()}, carpeta / "sttn.pth")     # mismo formato que gen_mejor.pth


def entrenar_conjunto():
    nombre = f"E_{MASCARA}_s{SEMILLA}"
    salida, frames_e = MODELOS / nombre, f"rgb_e2e_anterior_{MASCARA}_s{SEMILLA}"
    if terminado(nombre) and all(existe(v, frames_e) for v in DIVISION["validacion"] + DIVISION["prueba"]):
        print(f"{nombre}: ya entrenado, se reutiliza"); return
    if not terminado(nombre):
        gen = fijar_semilla(SEMILLA)
        datos = Clips(DIVISION["entrenamiento"], MASCARA)
        carga = torch.utils.data.DataLoader(datos, batch_size=LOTE_E, shuffle=True, drop_last=True, generator=gen)
        sttn = cargar_endo(PESOS_ENDO[MASCARA])
        if STTN_CAPAS == "transformer":
            for p in [*sttn.encoder.parameters(), *sttn.decoder.parameters()]:
                p.requires_grad = False
        enc, dec = cargar_estimador(PESOS_BASE)
        opt = torch.optim.Adam([
            {"params": enc.parameters(), "lr": LR / 10},
            {"params": dec.parameters(), "lr": LR},
            {"params": [p for p in sttn.parameters() if p.requires_grad], "lr": LR_STTN, "betas": (0.0, 0.99)},
        ])
        historial = {"mascara": MASCARA, "pesos_endo": str(PESOS_ENDO[MASCARA]), "semilla": SEMILLA,
                     "sttn_capas": STTN_CAPAS, "peso_hueco": PESO_HUECO, "peso_ancla": PESO_ANCLA,
                     "lote": LOTE_E, "lr": LR, "lr_sttn": LR_STTN, "division": DIVISION, "epocas": []}
        mejor, cambio = validar_conjunto(sttn, enc, dec)
        historial["epocas"].append({"epoca": 0, "abs_rel_val": mejor, "cambio_relleno_val": cambio})
        guardar_conjunto(sttn, enc, dec, salida)
        print(f"{nombre} ({len(datos)} clips de 5 frames) · época 0: AbsRel validación {mejor:.4f}")
        for ep in range(1, EPOCAS + 1):
            t0, suma = time.time(), {"prof": [], "hueco": [], "ancla": [], "total": []}
            sttn.train(); enc.train(); dec.train()
            for x, limpio, m, gt, brillo in carga:
                x, limpio, m, gt, brillo = (t.to(DEVICE) for t in (x, limpio, m, gt, brillo))
                pseudo = desplazar(m) * (1 - m)                       # hueco sobre tejido sano
                relleno = rellenar(sttn, x, (m + pseudo).clamp(max=1))
                x_limpio = x * (1 - m) + relleno * m                   # solo se pega en los reflejos
                a, b = brillo[:, 0].view(-1, 1, 1, 1, 1), brillo[:, 1].view(-1, 1, 1, 1, 1)
                entrada = (x_limpio * a + b).clamp(0, 1).flatten(0, 1)
                l_prof = perdida_si_log(predecir(enc, dec, entrada), gt.flatten(0, 1))
                l_hueco = l1_en(relleno, x, pseudo)
                l_ancla = l1_en(relleno, limpio, m)
                loss = l_prof + PESO_HUECO * l_hueco + PESO_ANCLA * l_ancla
                opt.zero_grad(); loss.backward(); opt.step()
                for k, v in zip(suma, (l_prof, l_hueco, l_ancla, loss)):
                    suma[k].append(v.item())
            val, cambio = validar_conjunto(sttn, enc, dec)
            fila = {"epoca": ep, **{f"perdida_{k}": float(np.mean(v)) for k, v in suma.items()},
                    "abs_rel_val": val, "cambio_relleno_val": cambio}
            historial["epocas"].append(fila)
            marca = ""
            if val < mejor:
                mejor, marca = val, "  <- mejor, guardado"
                guardar_conjunto(sttn, enc, dec, salida)
            print(f"  época {ep}: prof {fila['perdida_prof']:.4f} | hueco {fila['perdida_hueco']:.4f} | "
                  f"AbsRel validación {val:.4f} | cambio del relleno {cambio:.1f}/255 | {time.time() - t0:.0f} s{marca}")
            json.dump(historial, open(salida / "historial.json", "w"), indent=2)
        historial["mejor_abs_rel_val"] = mejor
        json.dump(historial, open(salida / "historial.json", "w"), indent=2)
        del sttn, enc, dec, opt; torch.cuda.empty_cache()

    # Frames limpiados por el Endo-STTN ajustado (mejor época), para evaluar E
    sttn = cargar_endo(salida / "sttn.pth")
    for v in DIVISION["validacion"] + DIVISION["prueba"]:
        guardar(v, frames_e, limpiar_video(sttn, cargar(v, "rgb"), cargar(v, f"mask_{MASCARA}")))
    print(f"{nombre}: listo ->", salida)

In [ ]:
for SEMILLA in SEMILLAS:
    print(f"\n========== Semilla {SEMILLA} ==========")
    if ENTRENAR["C"]:
        entrenar_estimador(f"C_s{SEMILLA}", "rgb")
    if ENTRENAR["D"]:
        entrenar_estimador(f"D_{MASCARA}_s{SEMILLA}", f"rgb_limpio_{MASCARA}")
    if ENTRENAR["E"]:
        entrenar_conjunto()

## 7. Prueba de profundidad con intervalos de confianza

Todos los modelos se reconstruyen a partir de sus pesos y se aplican a los videos de prueba, como en la revisión del
Avance 6: *median scaling* por frame; AbsRel, Sq Rel, RMSE, RMSE log y δ < 1.25; imagen completa y zonas con reflejo;
intervalos del 95 % por *bootstrap* de bloques de frames consecutivos, promediando las semillas disponibles.

Los modelos del Avance 5 (C, D y E) se buscan en `CARPETAS_NUESTRAS` con las mismas semillas de `SEMILLAS`.

In [ ]:
def buscar(nombre, carpetas):
    for c in carpetas:
        f = Path(c) / nombre / "historial.json"
        if f.exists() and "mejor_abs_rel_val" in json.load(open(f)):
            return Path(c) / nombre
    return None


etq = {"meslouhi": "El Meslouhi", "arnold": "Arnold"}[MASCARA]
MODELOS_EVAL = {"A · SCARED": [("A", PESOS_SCARED)], "A′ · equipo anterior": [("A", PESOS_ANT)]}
for t, nombre in (("C", "C_s{s}"), ("D", f"D_{MASCARA}_s{{s}}"), ("E", f"E_{MASCARA}_s{{s}}")):
    sufijo = "sin limpiar" if t == "C" else etq
    MODELOS_EVAL[f"{t} · {sufijo} (SCARED)"] = [(t, x) for s in SEMILLAS if (x := buscar(nombre.format(s=s), CARPETAS_NUESTRAS))]
    MODELOS_EVAL[f"{t}′ · {sufijo} (equipo anterior)"] = [(t, x) for s in SEMILLAS if (x := buscar(nombre.format(s=s), [MODELOS]))]
MODELOS_EVAL = {k: v for k, v in MODELOS_EVAL.items() if v}
display(pd.DataFrame([{"modelo": k, "semillas": len(v)} for k, v in MODELOS_EVAL.items()]).set_index("modelo"))

METRICAS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "a1"]
limpios_D = {v: None for v in VIDEOS_PRUEBA}
filas = []
for nombre, corridas in MODELOS_EVAL.items():
    for k, (tipo, carpeta) in enumerate(corridas):
        for v in VIDEOS_PRUEBA:
            rgb = cargar(v, "rgb")
            if tipo == "D":
                if limpios_D[v] is None:
                    sttn = cargar_endo(PESOS_ENDO[MASCARA]); limpios_D[v] = limpiar_video(sttn, rgb, cargar(v, f"mask_{MASCARA}")); del sttn
                entrada = limpios_D[v]
            elif tipo == "E":
                sttn = cargar_endo(carpeta / "sttn.pth"); entrada = limpiar_video(sttn, rgb, cargar(v, f"mask_{MASCARA}")); del sttn
            else:
                entrada = rgb
            enc, dec = cargar_estimador(carpeta)
            pred = predecir_video(enc, dec, entrada); del enc, dec
            prof = cargar(v, "depth")
            for i in range(len(pred)):
                g = np.asarray(prof[i], np.float32)
                for region, reg in (("imagen", None), ("reflejos", UNION[v][i])):
                    if (r := metricas(pred[i], g, reg)):
                        filas.append({"modelo": nombre, "corrida": k, "video": v, "frame": i, "region": region, **r})
        torch.cuda.empty_cache()
    print(f"{nombre}: {len(corridas)} semilla(s) evaluada(s)", flush=True)

PROF = pd.DataFrame(filas)
PROF.to_csv(RES / "profundidad_por_frame.csv", index=False)

In [ ]:
rng = np.random.default_rng(0)
CLAVES = pd.MultiIndex.from_tuples([(v, i) for v in VIDEOS_PRUEBA for i in range(len(cargar(v, "rgb")))], names=["video", "frame"])


def muestras_bloques(n_por_video, largo, B):
    salida = []
    for _ in range(B):
        idx, base = [], 0
        for n in n_por_video:
            inicios = rng.integers(0, max(1, n - largo + 1), size=int(np.ceil(n / largo)))
            idx.append((base + (inicios[:, None] + np.arange(largo)).ravel()[:n]).clip(max=base + n - 1))
            base += n
        salida.append(np.concatenate(idx))
    return np.array(salida)


def matriz(df, modelo, region, metrica):
    d = df[(df.modelo == modelo) & (df.region == region)]
    return d.pivot_table(index="corrida", columns=["video", "frame"], values=metrica).reindex(columns=CLAVES).to_numpy()


IDX = muestras_bloques([len(cargar(v, "rgb")) for v in VIDEOS_PRUEBA], LARGO_BLOQUE, N_BOOTSTRAP)


def estimar(M):
    por_frame = np.nanmean(M, axis=0)
    boot = np.nanmean(por_frame[IDX], axis=1)
    return float(np.nanmean(por_frame)), *np.nanpercentile(boot, [2.5, 97.5])


ENCABEZADO = {"abs_rel": "Abs Rel ↓", "sq_rel": "Sq Rel ↓", "rmse": "RMSE (mm) ↓", "rmse_log": "RMSE log ↓", "a1": "δ < 1.25 ↑"}
filas = []
for region in ("imagen", "reflejos"):
    for modelo in MODELOS_EVAL:
        fila = {"región": region, "modelo": modelo, "semillas": len(MODELOS_EVAL[modelo])}
        for met in METRICAS:
            fila[met], fila[f"{met}_lo"], fila[f"{met}_hi"] = estimar(matriz(PROF, modelo, region, met))
        filas.append(fila)
IC = pd.DataFrame(filas)
IC.to_csv(RES / "profundidad_ic95.csv", index=False)
fmt = lambda r, m: f"{r[m]:.3f} [{r[m + '_lo']:.3f}, {r[m + '_hi']:.3f}]"
for region in ("imagen", "reflejos"):
    t = IC[IC["región"] == region]
    display(Markdown(f"**Prueba de profundidad · {region}** (media [IC 95 %])"))
    display(pd.DataFrame({"Modelo": t.modelo, "Semillas": t.semillas,
                          **{ENCABEZADO[m]: [fmt(r, m) for _, r in t.iterrows()] for m in METRICAS}}).set_index("Modelo"))

# Diferencias pareadas: ¿partir de los pesos del equipo anterior cambia algo?
par = lambda t: (f"{t}′ · {'sin limpiar' if t == 'C' else etq} (equipo anterior)", f"{t} · {'sin limpiar' if t == 'C' else etq} (SCARED)")
COMPARACIONES = [("A′ · equipo anterior", "A · SCARED"), par("C"), par("D"), par("E"),
                 (par("E")[0], par("D")[0]), (par("E")[1], par("D")[1])]
filas = []
for region in ("imagen", "reflejos"):
    for a, b in COMPARACIONES:
        if a not in MODELOS_EVAL or b not in MODELOS_EVAL:
            continue
        for met in ("abs_rel", "rmse", "a1"):
            dif = np.nanmean(matriz(PROF, a, region, met), axis=0) - np.nanmean(matriz(PROF, b, region, met), axis=0)
            lo, hi = np.nanpercentile(np.nanmean(dif[IDX], axis=1), [2.5, 97.5])
            res = "sin diferencia" if lo <= 0 <= hi else f"mejor {a if (hi < 0) == (met != 'a1') else b}"
            filas.append({"región": region, "comparación": f"{a} − {b}", "métrica": ENCABEZADO[met],
                          "diferencia": round(float(np.nanmean(dif)), 4), "IC 95 %": f"[{lo:.4f}, {hi:.4f}]", "resultado": res})
DIFS = pd.DataFrame(filas)
DIFS.to_csv(RES / "diferencias_ic95.csv", index=False)
display(Markdown("**Diferencias pareadas con IC 95 %** (si el intervalo no incluye 0, la diferencia es significativa)"))
display(DIFS.set_index(["región", "comparación", "métrica"]))

# Tabla en LaTeX (imagen completa)
t = IC[IC["región"] == "imagen"]
lineas = [r"\begin{tabular}{l" + "c" * len(METRICAS) + "}", r"\toprule",
          "Modelo & " + " & ".join([r"Abs Rel $\downarrow$", r"Sq Rel $\downarrow$", r"RMSE (mm) $\downarrow$",
                                    r"RMSE log $\downarrow$", r"$\delta<1.25$ $\uparrow$"]) + r" \\", r"\midrule"]
lineas += [f"{r.modelo} & " + " & ".join(fmt(r, m) for m in METRICAS) + r" \\" for _, r in t.iterrows()]
lineas += [r"\bottomrule", r"\end{tabular}"]
open(RES / "tabla_pesos_anterior_ic95.tex", "w").write("\n".join(lineas) + "\n")
print("Tabla LaTeX:", RES / "tabla_pesos_anterior_ic95.tex")

In [ ]:
# Gráfica: AbsRel con IC 95 % (imagen completa), nuestros modelos frente a los que parten del equipo anterior
t = IC[(IC["región"] == "imagen")].reset_index(drop=True)
color = lambda n: "#2f855a" if "anterior" in n else "#2a78d6"
fig, ax = plt.subplots(figsize=(9, 0.6 * len(t) + 1.2))
for k, r in t.iterrows():
    ax.errorbar(r.abs_rel, k, xerr=[[r.abs_rel - r.abs_rel_lo], [r.abs_rel_hi - r.abs_rel]], fmt="o", ms=7,
                color=color(r.modelo), capsize=4, lw=2)
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.modelo); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlabel("AbsRel (escala logarítmica; menor es mejor)")
ax.set_title("AbsRel en prueba con IC 95 % · azul: parte de SCARED · verde: parte del equipo anterior")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.savefig(RES / "pesos_anterior_ic95.png", dpi=100, bbox_inches="tight"); plt.show()
print("Resultados en", RES)

## 8. Si los pesos traen un Endo-STTN: calidad del relleno

Solo se ejecuta si en la sección 2 apareció un Endo-STTN. Compara su relleno con el nuestro (reentrenado en el
Avance 4) en el **pseudo-hueco** de los videos de prueba, donde se conoce el color real, con el mismo protocolo de la
revisión del Avance 6.

In [ ]:
from skimage.metrics import structural_similarity


def hueco_np(mask):
    h = np.zeros_like(mask)
    h[..., DESPLAZAMIENTO:] = mask[..., :-DESPLAZAMIENTO]
    return h & ~mask


def metricas_hueco(pred, real, hueco, minimo=30):
    if hueco.sum() < minimo:
        return None
    d = pred[hueco].astype(np.float64) - real[hueco].astype(np.float64)
    _, mapa = structural_similarity(real, pred, channel_axis=2, data_range=255, full=True)
    return {"mae": float(np.abs(d).mean()), "psnr": float(10 * np.log10(255 ** 2 / max(float((d ** 2).mean()), 1e-10))),
            "ssim": float(mapa.mean(-1)[hueco].mean())}


if STTN_ANT is None:
    print("Los pesos del equipo anterior no traen un Endo-STTN: se omite esta sección.")
else:
    filas = []
    for nombre, pesos in ((f"Avance 4 · {etq} (nuestro)", PESOS_ENDO[MASCARA]), ("Equipo anterior", STTN_ANT)):
        sttn = cargar_endo(pesos)
        for v in VIDEOS_PRUEBA:
            rgb, mask = np.asarray(cargar(v, "rgb")), np.asarray(cargar(v, "mask_meslouhi"))
            hueco = hueco_np(mask)
            lleno = limpiar_video(sttn, rgb, hueco)
            filas += [{"Endo-STTN": nombre, **r} for i in range(len(rgb)) if (r := metricas_hueco(lleno[i], rgb[i], hueco[i]))]
        del sttn; torch.cuda.empty_cache()
    RELLENO = pd.DataFrame(filas)
    RELLENO.to_csv(RES / "relleno_sttn_anterior.csv", index=False)
    display(Markdown("**Relleno en el pseudo-hueco de prueba** (PSNR y SSIM: mayor es mejor; MAE: menor es mejor)"))
    display(RELLENO.groupby("Endo-STTN")[["psnr", "ssim", "mae"]].mean().round(3))

## 9. Cómo leer los resultados

- **Sección 2:** si los pesos no son AF-SfMLearner (ResNet-18 o 50), el notebook se detiene ahí: hay que adaptar el
  estimador a su arquitectura.
- **Sección 4:** si aparece **FUGA**, los resultados de A′, C′, D′ y E′ no son válidos en esta prueba. Si la fuga es
  **desconocida**, repórtalos con esa advertencia.
- **A′ frente a A:** dice cuál es mejor punto de partida *sin* ajuste. Un A′ mucho mejor que A (0.396) sugiere que el
  equipo anterior ya ajustó a C3VD: revisa con más cuidado la sección 4.
- **C′, D′, E′ frente a C, D, E:** dice si partir de sus pesos mejora el ajuste fino. Con 225 frames de entrenamiento,
  el punto de partida puede pesar bastante.
- **E′ − D′ frente a E − D:** es la comparación más útil para el artículo. Si E′ también mejora a D′ (intervalo de la
  diferencia por debajo de 0), la ventaja del entrenamiento conjunto **no depende** de partir de SCARED.
- Como en la revisión, los intervalos reflejan la variación entre frames de un solo segmento de prueba, no entre
  semillas ni entre segmentos.